## This notebook is made to cover the practical part of the Machine Learning project that is related to creating a classification prediction model

Before introducing any kind of information in the dataset and/or features, the dataset **MUST BE SPLITTED**

Since we are dealing with a small unbalances dataset 4424 records, the split is going to be 80-20 with K-fold cross-validation to avoid problems with the target column since that is the feature we want to predict. The number of folds we do in the train data could also be changed to see which result we can get out of this parameter, first we are going to start with k = 10

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

raw_dataframe = pd.read_csv('dropout_dataset.csv', sep=';')


#Before really splitting the data, some cleaning is done to avoid annoying errors in the future

for column in raw_dataframe.columns:
    raw_dataframe.rename(columns = {f'{column}':f'{column.rstrip().lstrip()}'})



X = raw_dataframe.drop(columns=["Target"])
y = raw_dataframe["Target"]



X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2, random_state=42)

##Check if there is any NaN values in the datasets

print("NaN values on X_train before preprocessing:\n", X_train.isna().sum())
print(X_train.info())
#print("NaN values on X_test before preprocessing:\n", X_test.isna().sum())
#print("NaN values on Y_train before preprocessing:\n", Y_train.isna().sum())
#print("NaN values on Y_test before preprocessing:\n", Y_test.isna().sum())


NaN values on X_train before preprocessing:
 Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship holder 

Now that the split have been done, we need to define a strategy to deal with the imbalanced aspect of the dataset. In the reference paper, they used SMOTE, ADASYN and Logistic regression, regarding the avaliable resampling methods, we can try to apply 
- Oversampling techniques
  1. Borderline-SMOTE
  2. random over-sampling
  3. SMOTE-ENN
  4. SMOTENC (especifically designed for nominal and continuous features)

OR

- Algorithmic Solutions
  1. Weighted loss function
  2. Classical weight adjustment
  3. Focal loss


Since our dataset is small, there will be a very big loss of information if we do under-sampling because reducing even more the dataset will pose even more problems, the oversampling techniques will be used for this case

Regarding the algorithms to build the models, since our task is to classificate or fit into classification the three categories of students based on the given existing features:
- Success
- Relative success
- Failure

And knowing that in the reference study they used **Logistic regression, SVM, decision tree, random forest, Gradient boosting, Xtreme gradient boosting, legit boost and cat boost**. Some other avaliable options for algorithms are:
- **Probabilistic & Linear Models**
    1. Naïve Bayes (NB)
- **Neural Networks**
    1. Multi-Layer Perceptron (MLP - Feedforward Neural Network)
- **Rule-Based & Distance-Based Models**
    1. K-Nearest Neighbors (KNN)
- **Ensemble & Evolutionary Methods**
    1. Voting classifier
    2. Genetic algorithms

**1. First part of the study: For each model (or one selected model), we can test different sample spaces -> 4 x N_models** 
This will give the output of which sampling method should be used in the study

**2. Second part of the study: for each model, which one can give more satisfactory results -> N_models**
This will give the output of which model is more appropriate for this task

**3. For the most appropriate model, what k-fold is ideal for getting better results(evaluate if it makes sense)**

**4. Compare the results with the study results and draw conclusions**

## Which sample strategy yields the best result?

To answer this question, we will take a similar approach to the reference paper, we pick first a classification method and then run that method with the different sample spaces that we have and then compare the different performances of them.
Taking into consideration our dataset carachteristics and the goal of the model (correctly classify the most under-represented class in the sample set "Partial success/enrolled"), the most appropriated ones are the ones that evaluates how well the classification is done to the most crititcal class, with that in mind, the most appropriate evaluation method is the F1 score (the same one used in the paper). This also keeps the comparision fair so we don't end up in the case of comparing apples to oranges.

The naive bayes method is the one that would require the biggest dataset treatment before applying the model and the KNN is one model that would suffer from the curse of dimensionality, since on this step, the goal is to evaluate which oversampling method is the best, we are going to proceed with the model that presents the most straight forward setup to this evaluation MLP.

To prepare our MLP model, these are the preparations that we need to do:
1. Normalize the continuous features
2. Normalize discrete numerical features
3. One-hot encode the categorical features
4. Normalize the ordinal features

#### 1.Normalize continuous features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

**First we treat the training set, and then the test set**

In [2]:
from sklearn.preprocessing import MinMaxScaler

continuous_min_max_scaler = MinMaxScaler()

X_train_treatment = X_train.copy()

df_continuous_columns = ['Previous qualification (grade)','Admission grade', 'Unemployment rate', 
    'Inflation rate','GDP', 'Curricular units 1st sem (grade)','Curricular units 2nd sem (grade)']

df_continuous_columns_data = X_train_treatment[df_continuous_columns]

print(X_train_treatment[df_continuous_columns].describe())

df_continuous_columns_normalized = continuous_min_max_scaler.fit_transform(df_continuous_columns_data)

df_continuous_columns_normalized = pd.DataFrame(
    df_continuous_columns_normalized, 
    columns=df_continuous_columns,
    index=X_train_treatment.index
)

X_train_treatment[df_continuous_columns] = df_continuous_columns_normalized

print("Dimension of the train dataset with continuous normalization = {}".format(X_train_treatment.shape))

       Previous qualification (grade)  Admission grade  Unemployment rate  \
count                     3539.000000      3539.000000        3539.000000   
mean                       132.649392       126.851936          11.563182   
std                         13.239906        14.485637           2.669147   
min                         95.000000        95.000000           7.600000   
25%                        125.000000       117.800000           9.400000   
50%                        133.100000       126.100000          11.100000   
75%                        140.000000       134.700000          13.900000   
max                        190.000000       190.000000          16.200000   

       Inflation rate          GDP  Curricular units 1st sem (grade)  \
count     3539.000000  3539.000000                       3539.000000   
mean         1.236253     0.008918                         10.626387   
std          1.379311     2.273285                          4.848718   
min         -0.800

In [3]:
##NaN values in the first treatment

print("NaN values before preprocessing:\n", X_train_treatment.isna().sum())

NaN values before preprocessing:
 Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship holder            

In [4]:
X_test_treated = X_test.copy()  # Copy the test data

# Select the continuous columns from the test dataset
df_continuous_columns_data_test = X_test_treated[df_continuous_columns]

# Use the existing scaler to transform the test data (do not fit, only transform)
df_continuous_columns_normalized_test = continuous_min_max_scaler.transform(df_continuous_columns_data_test)

# Convert the normalized data back to a DataFrame and preserve column names
df_continuous_columns_normalized_test = pd.DataFrame(df_continuous_columns_normalized_test, columns=df_continuous_columns,index=X_test_treated.index)

# Replace the original continuous columns in the test data with the normalized ones
X_test_treated[df_continuous_columns] = df_continuous_columns_normalized_test

print("Dimension of the test dataset with continuous normalization = {}".format(X_test_treated.shape))

Dimension of the test dataset with continuous normalization = (885, 36)


In [5]:
##NaN values in the first treatment

print("NaN values before preprocessing:\n", X_test_treated.isna().sum())

NaN values before preprocessing:
 Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship holder            

#### 2.Normalize discrete numerical features
To normalize them, we will use min-max scaling (we are aware that this causes loss of information)

In [6]:
df_discrete_columns =['Age at enrollment', 'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)',
     'Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)',
     'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (enrolled)',
     'Curricular units 2nd sem (evaluations)','Curricular units 2nd sem (approved)',
     'Curricular units 2nd sem (without evaluations)'
    ]

for column in df_discrete_columns:
    min_val = X_train_treatment[column].min()
    max_val = X_train_treatment[column].max()
    print(f"Column: {column}")
    print(f"Min: {min_val}, Max: {max_val}")
    print("-" * 50)

print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with this treatment = {}".format(X_train_treatment.shape))

Column: Age at enrollment
Min: 17, Max: 70
--------------------------------------------------
Column: Curricular units 1st sem (credited)
Min: 0, Max: 20
--------------------------------------------------
Column: Curricular units 1st sem (enrolled)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (evaluations)
Min: 0, Max: 45
--------------------------------------------------
Column: Curricular units 1st sem (approved)
Min: 0, Max: 26
--------------------------------------------------
Column: Curricular units 1st sem (without evaluations)
Min: 0, Max: 12
--------------------------------------------------
Column: Curricular units 2nd sem (enrolled)
Min: 0, Max: 23
--------------------------------------------------
Column: Curricular units 2nd sem (evaluations)
Min: 0, Max: 33
--------------------------------------------------
Column: Curricular units 2nd sem (approved)
Min: 0, Max: 20
--------------------------------------------------
C

Since our discrete columns have a fairly significant range, we will normalize them

In [7]:
discrete_scaler = MinMaxScaler()

df_discrete_columns_data = X_train_treatment[df_discrete_columns]

df_discrete_columns_normalized = discrete_scaler.fit_transform(df_discrete_columns_data)

df_discrete_columns_normalized = pd.DataFrame(df_discrete_columns_normalized, 
                                              columns=df_discrete_columns,
                                              index=X_train_treatment.index
                                             )

X_train_treatment[df_discrete_columns] = df_discrete_columns_normalized

print("Dimension of the train dataset with discrete treatment = {}".format(X_train_treatment.shape))


print("NaN values before preprocessing:\n", X_train_treatment.isna().sum())

Dimension of the train dataset with discrete treatment = (3539, 36)
NaN values before preprocessing:
 Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender        

In [8]:
# Select discrete columns from the test dataset
df_discrete_columns_data_test = X_test_treated[df_discrete_columns]

# Use the existing scaler to transform the test data (do not fit, only transform)
df_discrete_columns_normalized_test = discrete_scaler.transform(df_discrete_columns_data_test)

# Convert the normalized data back to a DataFrame and preserve column names
df_discrete_columns_normalized_test = pd.DataFrame(df_discrete_columns_normalized_test, 
                                                   columns=df_discrete_columns,
                                                   index=X_test_treated.index
                                                  )

# Replace the original discrete columns in the test data with the normalized ones
X_test_treated[df_discrete_columns] = df_discrete_columns_normalized_test

print("Dimension of the test dataset with discrete treatment = {}".format(X_test_treated.shape))

print("NaN values before preprocessing:\n", X_test_treated.isna().sum())

Dimension of the test dataset with discrete treatment = (885, 36)
NaN values before preprocessing:
 Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender          

#### 3.One hot encode the categorical features

After normalizing the data in hand we can one hot encode all the categorical feature, this technique transform all the categories in a separate column (known as dummy column) where the values are binary, with that transformation, the data related to these columns is more digestable to the algorithm

In [9]:
from sklearn.preprocessing import OneHotEncoder

df_categorical_columns = ['Application mode', 'Course', 'Previous qualification', 'Nacionality',"Father's occupation", "Mother's occupation"]

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

one_hot_encoded_categorical = encoder.fit_transform(X_train_treatment[df_categorical_columns])

# Fix: Ensure the new DataFrame retains the original index
encoded_df = pd.DataFrame(one_hot_encoded_categorical, 
                          columns=encoder.get_feature_names_out(df_categorical_columns),
                          ##Prevents pd to create rows to fit the new/unsetted index
                          ##If this is not explicitly done, pandas try to create new rows
                          ## to fit the mismatched indexes. This can introduce unwanted noise into our
                          ## sample space
                          index=X_train_treatment.index)


X_train_treatment = pd.concat([X_train_treatment.drop(columns=df_categorical_columns), encoded_df], axis=1)

print("Dimension of the train dataset with categorical treatment = {}".format(X_train_treatment.shape))


Dimension of the train dataset with categorical treatment = (3539, 172)


In [10]:
# Transform categorical columns in the test data (do not fit, only transform)
one_hot_encoded_categorical_test = encoder.transform(X_test_treated[df_categorical_columns])

# Convert the encoded test data into a DataFrame while preserving column names and index
encoded_df_test = pd.DataFrame(one_hot_encoded_categorical_test, 
                               columns=encoder.get_feature_names_out(df_categorical_columns), 
                               index=X_test_treated.index)

# Replace categorical columns in the test data with the encoded ones
X_test_treated = pd.concat([X_test_treated.drop(columns=df_categorical_columns), encoded_df_test], axis=1)


print("Dimension of the test dataset with categorical treatment = {}".format(X_test_treated.shape))

Dimension of the test dataset with categorical treatment = (885, 172)


/home/samuelaga/Documents/Master2IS/ArtificialIntelligence/jupyterenv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


#### 4. Normalize the ordinal features
Since for ordinal features the order is embedded on the distance between the values, we can just apply the regular normalization to avoid dimensionality problems with them

**On this case, we are considering that the loss of information in treating ordinal as categorical can prejudice the model somehow, this can be also another test to make**

Taking a closer look on the values of the preassumed ordinal features, we can see that the numbers **do not** hold any order or meaning e.g: 
- Can't read or write is 35, but Higher Education - Bachelor's Degree is 2, which is completely out of order.
- 12th Year of Schooling - Not Completed is 9, but 7th Year (Old) is 11, which doesn’t make sense in an ascending order.
- Can't read or write is 35, but Higher Education - Doctorate (3rd cycle) is 44.
- Basic education 1st cycle (4th/5th year) is 37, while Higher Education - Bachelor's Degree is 2.

With this observation we can choose two paths, treat them as categorical and one hot encode them or create groups to logically order them, the first path use the data as is and does not give any more meaning to it but the second one can enrich the data but the impact on the quality of the classification is unkown.

Since in general, scholarity level is considered ordinal, that is the approach we are going to take on this model

In [11]:
macro_groups_of_higher_education = {
    'No formal Education': 1,
    'Primary Education (Basic Education - 1st Cycle)': 2,
    'Lower Secondary Education (Basic Education - 2nd Cycle)':3,
    'Upper Secondary Education (Basic Education - 3rd Cycle)':4,
    'Technical & Professional Education': 5,
    'Higher Education (University Level)':6,
    'Unknown':7
}

education_level_mapping_father_qualification = {
    35: 1,
    36: 1,
    37: 2,
    38: 3,
    26: 3,
    11: 3,
    30: 3,
    19: 4,
    29: 4,
    14: 4,
    12: 4,
    10: 4,
    9: 4,
    1: 4,
    22: 5,
    31: 5,
    18: 5,
    33: 5,
    25: 5,
    27: 5,
    20: 5,
    13: 5,
    39: 5,
    6: 6,
    2: 6,
    3: 6,
    40: 6,
    41: 6,
    42: 6,
    4: 6,
    43: 6,
    5: 6,
    44: 6,
    34: 7
}

##There is a difference on the qualification mapping

education_level_mapping_mother_qualification = {
    1: 4,
    2: 6,
    3: 6,
    4: 6,
    5: 6,
    6 :6,
    9: 4,
    10: 4,
    11: 3,
    12: 4,
    14: 4,
    18: 5,
    19: 4,
    22: 5,
    26: 3,
    27: 5,
    29: 4,
    30: 3,
    34: 7,
    35: 1,
    36: 1,
    37: 2,
    38: 3,
    39: 5,
    40: 6,
    41: 6,
    42: 6,
    43: 6,
    44: 6,
}

X_train_treatment["Father's qualification"] = X_train_treatment["Father's qualification"].map(education_level_mapping_father_qualification)

X_train_treatment["Mother's qualification"] = X_train_treatment["Mother's qualification"].map(education_level_mapping_father_qualification)

X_test_treated["Father's qualification"] = X_test_treated["Father's qualification"].map(education_level_mapping_father_qualification)

X_test_treated["Mother's qualification"] = X_test_treated["Mother's qualification"].map(education_level_mapping_father_qualification)


After logically mapping them, we can finally normalize the values

In [12]:
ordinal_scaler = MinMaxScaler()

df_ordinal_columns = ['Marital status',  'Application order', "Mother's qualification", "Father's qualification"]

df_ordinal_columns_data = X_train_treatment[df_ordinal_columns]

df_ordinal_columns_normalized = ordinal_scaler.fit_transform(df_ordinal_columns_data)

df_ordinal_columns_normalized = pd.DataFrame(df_ordinal_columns_normalized, columns=df_ordinal_columns, index=X_train_treatment.index)

X_train_treatment[df_ordinal_columns] = df_ordinal_columns_normalized

print("Dimension of the train dataset with ordinal treatment = {}".format(X_train_treatment.shape))



Dimension of the train dataset with ordinal treatment = (3539, 172)


In [13]:
df_ordinal_columns_data_test = X_test_treated[df_ordinal_columns]

df_ordinal_columns_normalized_test = ordinal_scaler.transform(df_ordinal_columns_data_test)

# Convert the transformed data into a DataFrame while preserving column names and index
df_ordinal_columns_normalized_test = pd.DataFrame(df_ordinal_columns_normalized_test, 
                                                  columns=df_ordinal_columns, 
                                                  index=X_test_treated.index)

# Replace ordinal columns in the test data with the normalized ones
X_test_treated[df_ordinal_columns] = df_ordinal_columns_normalized_test

print("Dimension of the test dataset with ordinal treatment = {}".format(X_test_treated.shape))

Dimension of the test dataset with ordinal treatment = (885, 172)


Now that our dataset is treated and ready to be ingested by the model we can proceed to the next steps. Just for comparision lets check the difference in dimension for the treated datased and the original one

In [14]:
print("##################### Train dataset")
print("Dimension of the train dataset without the treatments = {}".format(X_train.shape))
print("Dimension of the train dataset with the treatments = {}".format(X_train_treatment.shape))

print("##################### test dataset")
print("Dimension of the test dataset without the treatments = {}".format(X_test.shape))
print("Dimension of the test dataset with the treatments = {}".format(X_test_treated.shape))

##################### Train dataset
Dimension of the train dataset without the treatments = (3539, 36)
Dimension of the train dataset with the treatments = (3539, 172)
##################### test dataset
Dimension of the test dataset without the treatments = (885, 36)
Dimension of the test dataset with the treatments = (885, 172)


In [15]:
# ##Check NaN values
# print("##################### Train dataset")
# #print("NaN values in X_train= {}".format(X_train_treatment.isnull().any()))
# print("NaN values in X_train= {}".format(X_train_treatment.columns[X_train_treatment.isna().any()].tolist()))
# #print("NaN values in Y_train = {}".format(Y_train.isnull().any()))
# print("NaN values in Y_train= {}".format(Y_train.columns[Y_train.isna().any()].tolist()))

# print("##################### test dataset")
# print("NaN values in X_test = {}".format(X_test_treated.isnull().any()))
# print("NaN values in Y_test = {}".format(Y_train.isnull().any()))

In [16]:
print("Xshape_train_dirty = {}".format(X_train_treatment.shape))
print("Yshape_train_dirty = {}".format(Y_train.shape))


merged_x_and_y_train = pd.concat([X_train_treatment, Y_train], axis=1)
#print("Dimension of the train dataset with the treatments = {}".format(merged_x_and_y_train.shape))

merged_x_and_y_train = merged_x_and_y_train.dropna()

#print("Dimension of the train dataset with the treatments = {}".format(merged_x_and_y_train.shape))

X_train_cleaned = merged_x_and_y_train.drop(columns=['Target'])
Y_train_cleaned = merged_x_and_y_train['Target']

print("X_train_cleaned = {}".format(X_train_cleaned.shape))
print("Y_train_cleaned = {}".format(Y_train_cleaned.shape))

Xshape_train_dirty = (3539, 172)
Yshape_train_dirty = (3539,)
X_train_cleaned = (3539, 172)
Y_train_cleaned = (3539,)


In [17]:
print("Xshape_test_dirty = {}".format(X_test_treated.shape))
print("Yshape_test_dirty = {}".format(Y_test.shape))


merged_x_and_y_test = pd.concat([X_test_treated, Y_test], axis=1)
#print("Dimension of the test dataset with the treatments = {}".format(merged_x_and_y_test.shape))

merged_x_and_y_test = merged_x_and_y_test.dropna()

#print("Dimension of the train dataset with the treatments = {}".format(merged_x_and_y_test.shape))

X_test_cleaned = merged_x_and_y_test.drop(columns=['Target'])
Y_test_cleaned = merged_x_and_y_test['Target']

print("X_train_cleaned = {}".format(X_test_cleaned.shape))
print("Y_train_cleaned = {}".format(Y_test_cleaned.shape))

Xshape_test_dirty = (885, 172)
Yshape_test_dirty = (885,)
X_train_cleaned = (885, 172)
Y_train_cleaned = (885,)


With this output we see that there are columns in train that are not present in test and columns in test that are not present in train. Since they are all one-hot encoded columns, there is no problem in setting the missing columns as zero. Also, we need to put the test columns in the same order since for MLP's the order matters for the final output

In [18]:
missing_in_train = set(X_test_treated.columns) - set(X_train_cleaned.columns)
missing_in_test = set(X_train_cleaned.columns) - set(X_test_treated.columns)

print("Columns in train but missing in test:", missing_in_test)
print("Columns in test but missing in train:", missing_in_train)


Columns in train but missing in test: set()
Columns in test but missing in train: set()


In [19]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
Y_train_cleaned = encoder.fit_transform(Y_train_cleaned)
Y_test_cleaned = encoder.transform(Y_test_cleaned)

for category, code in zip(encoder.classes_, range(len(encoder.classes_))):
    print(f"{category} -> {code}")

Dropout -> 0
Enrolled -> 1
Graduate -> 2


Now that the datasets have the same dimension and have been through the same preprocessing, we can proceed

## Comparing how well the MLP perform under different oversample techniques

### Create a sample space using borderline SMOTE

In [20]:
from collections import Counter
from imblearn.over_sampling import BorderlineSMOTE

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

sm = BorderlineSMOTE(random_state=42)
X_borderline_SMOTE, Y_borderline_SMOTE = sm.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_borderline_SMOTE))


Original training dataset shape Counter({np.int64(2): 1791, np.int64(0): 1105, np.int64(1): 643})
Resampled training dataset shape Counter({np.int64(0): 1791, np.int64(1): 1791, np.int64(2): 1791})


### Create a sample space using random over sampler

In [21]:
from imblearn.over_sampling import RandomOverSampler

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

ros = RandomOverSampler(random_state=42)
X_random_over_sampler, Y_random_over_sampler = ros.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_random_over_sampler))


Original training dataset shape Counter({np.int64(2): 1791, np.int64(0): 1105, np.int64(1): 643})
Resampled training dataset shape Counter({np.int64(0): 1791, np.int64(1): 1791, np.int64(2): 1791})


### Create a sample space using random over SMOTE EEN

Here, the result is a little different than those we saw in the other resample stretegies

In [22]:
from imblearn.combine import SMOTEENN

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

sme = SMOTEENN(random_state=42)
X_smote_een, Y_smote_een = sme.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_smote_een))


Original training dataset shape Counter({np.int64(2): 1791, np.int64(0): 1105, np.int64(1): 643})
Resampled training dataset shape Counter({np.int64(1): 1345, np.int64(0): 940, np.int64(2): 584})


**The difference in the result set might make this method unsuitabel, even though we managed to increase the number of enrolled targets we significantly decreased the other ones, with that, we just shifted the imbalance of the data to the other targets. To confirm this hypothesis we need to run the tests**

### Create a sample space using random over SMOTENC

Here, the result is a little different than those we saw in the other resample stretegies

In [23]:
from imblearn.over_sampling import SMOTENC

print('Original training dataset shape %s' % Counter(Y_train_cleaned))

smnc = SMOTENC(random_state=42, categorical_features= [1,3,5,7])
X_smote_smnc, Y_smote_smnc = smnc.fit_resample(X_train_cleaned,Y_train_cleaned)

print('Resampled training dataset shape %s' % Counter(Y_smote_smnc))

Original training dataset shape Counter({np.int64(2): 1791, np.int64(0): 1105, np.int64(1): 643})
Resampled training dataset shape Counter({np.int64(0): 1791, np.int64(1): 1791, np.int64(2): 1791})


### Now that the samplespaces are created, we need to create a model to using each one of those and evaluate their performance to select one of the methods

Even when applying the treatment in the test dataset we still dont get the same dimensionality on it to use, before comparing the models we need to fix this issue

In [24]:
from sklearn.neural_network import MLPClassifier


PLAIN_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_train_cleaned, Y_train_cleaned)
borderline_SMOTE_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_borderline_SMOTE, Y_borderline_SMOTE)
random_over_sampler_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_random_over_sampler, Y_random_over_sampler)
SMOTE_EEN_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_smote_een, Y_smote_een)
SMOTE_NC_MLP = MLPClassifier(random_state=1, max_iter=1000).fit(X_smote_smnc, Y_smote_smnc)

In [25]:
Y_PLAIN_MLP = PLAIN_MLP.predict(X_test_treated)
Y_pred_borderline_SMOTE_MLP = borderline_SMOTE_MLP.predict(X_test_treated)
Y_random_over_sampler_MLP = random_over_sampler_MLP.predict(X_test_treated)
Y_SMOTE_EEN_MLP = SMOTE_EEN_MLP.predict(X_test_treated)
Y_SMOTE_NC_MLP = SMOTE_NC_MLP.predict(X_test_treated)

After creating the predictions, we can finally evaluate the performance of the models and comparing them using F1

In [26]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report


PLAIN_MLP = f1_score(Y_test_cleaned, Y_PLAIN_MLP, average=None,zero_division=1.0)
F1_borderline_SMOTE_MLP = f1_score(Y_test_cleaned, Y_pred_borderline_SMOTE_MLP, average=None,zero_division=1.0)
F1_random_over_sampler_MLP = f1_score(Y_test_cleaned, Y_random_over_sampler_MLP, average=None,zero_division=1.0)
F1_SMOTE_EEN_MLP = f1_score(Y_test_cleaned, Y_SMOTE_EEN_MLP, average=None,zero_division=1.0)
F1_SMOTE_NC_MLP = f1_score(Y_test_cleaned, Y_SMOTE_NC_MLP, average= None,zero_division=1.0)

#Dropout -> 0
#Enrolled -> 1
#Graduate -> 2

target_names = ['Dropout','Enrolled', 'Graduate']


#print(F1_borderline_SMOTE_MLP)
#print(F1_random_over_sampler_MLP)
#print(F1_SMOTE_EEN_MLP)
#print(F1_SMOTE_NC_MLP)
print("\t plain MLP results ###########################\n")
print(classification_report(Y_test_cleaned, Y_PLAIN_MLP,target_names=target_names,zero_division=0.0))
print("\t borderline smote results ###########################\n")
print(classification_report(Y_test_cleaned, Y_pred_borderline_SMOTE_MLP,target_names=target_names,zero_division=0.0))
print("\t Random over sampler results ###########################\n")
print(classification_report(Y_test_cleaned, Y_random_over_sampler_MLP,target_names=target_names,zero_division=0.0))
print("\t SMOTE EEN results ###########################\n")
print(classification_report(Y_test_cleaned, Y_SMOTE_EEN_MLP,target_names=target_names,zero_division=0.0))
print("\t SMOTE NC results ###########################\n")
print(classification_report(Y_test_cleaned, Y_SMOTE_NC_MLP,target_names=target_names,zero_division=0.0))

	 plain MLP results ###########################

              precision    recall  f1-score   support

     Dropout       0.76      0.70      0.73       316
    Enrolled       0.38      0.39      0.39       151
    Graduate       0.77      0.82      0.80       418

    accuracy                           0.70       885
   macro avg       0.64      0.64      0.64       885
weighted avg       0.70      0.70      0.70       885

	 borderline smote results ###########################

              precision    recall  f1-score   support

     Dropout       0.77      0.71      0.74       316
    Enrolled       0.39      0.40      0.40       151
    Graduate       0.79      0.83      0.81       418

    accuracy                           0.71       885
   macro avg       0.65      0.65      0.65       885
weighted avg       0.71      0.71      0.71       885

	 Random over sampler results ###########################

              precision    recall  f1-score   support

     Dropout       